In [ ]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q

# Import des packages nécessaires

In [ ]:
import pandas as pd
import io
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.styles import Font, PatternFill,Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.series import SeriesLabel
from openpyxl.utils import get_column_letter
from openpyxl.chart.label import DataLabelList

# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [ ]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2022) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [ ]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [ ]:
df_effectifs[df_effectifs.duplicated(keep=False)]


#### Nombre de doublons par colonne

In [ ]:
df_effectifs.apply(lambda col: col.duplicated().sum())


### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [ ]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



In [ ]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [ ]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

In [ ]:
df_effectifs.head()


In [ ]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [ ]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [ ]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [ ]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [ ]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [ ]:
df_effectifs.dtypes


### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [ ]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [ ]:
df_effectifs.dtypes


In [ ]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [ ]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [ ]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [ ]:
df_effectifs = df_effectifs.dropna()

In [ ]:
df_effectifs

### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [ ]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [ ]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2022) &
        (chunk['montant'] > 0) 
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


# Les champs sont les suivants avant nettoyage:

In [ ]:
df_depenses.columns


In [ ]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [ ]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

In [ ]:
df_depenses = df_depenses.drop_duplicates()


In [ ]:
df_depenses


# Filtrage avant creation du dashboard

In [ ]:
df_depenses = df_depenses.drop('Total consommants tous régimes', errors='ignore')

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleanedData_depenses" et "cleanedData_Pathos"

Pour les dépenses et pour les pathologies :

In [ ]:
output_dep = '../DATA/Dashboard_Depenses.xlsx'

with pd.ExcelWriter(output_dep, engine='openpyxl') as writer:
    df_depenses.to_excel(writer, sheet_name='cleanedData_depenses', index=False)

print(f"Fichier dépenses créé : {output_dep}")


In [ ]:
output_eff = '../DATA/Dashboard_Effectifs.xlsx'

with pd.ExcelWriter(output_eff, engine='openpyxl') as writer:
    df_effectifs.to_excel(writer, sheet_name='cleanedData_Effectifs', index=False)

print(f"Fichier effectifs créé : {output_eff}")


In [ ]:
wb_eff = load_workbook(output_eff)

In [ ]:
wb_dep = load_workbook(output_dep) 

Calcul des listes _ len_dict

In [ ]:
annees = sorted(df_depenses['annee'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].dropna().unique())
postes = sorted(df_depenses['dep_niv_1'].dropna().unique())

# LISTES POUR FILTRES EFFECTIFS
depts = sorted(df_effectifs['Département'].dropna().unique())
classes_age = sorted(df_effectifs['Classe d\'age'].dropna().unique())
sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())
pathos_eff = sorted(df_effectifs['patho_niv1'].dropna().unique())

# LEN_DICT
len_dict = {
    'len_annee': len(annees) + 1,
    'len_patho_niv1': len(pathos) + 1,
    'len_dep_niv_1': len(postes) + 1,
    'len_departement_eff': len(depts) + 1,
    'len_classes_age': len(classes_age) + 1,
    'len_sexe': len(sexes) + 1,
    'len_region': len(regions) + 1,
    'len_patho_effectif': len(pathos_eff) + 1, 
}

import pprint
pprint.pprint(len_dict)

In [ ]:
cols_to_calculate = ['annee', 'patho_niv1', 'dep_niv_1', 'dep_niv_2']

cols_to_calculate_effectifs = {
    'annee_eff': 'annee',
    'patho_effectif': 'patho_niv1',
    'age': "Classe d'age",
    'sexe': 'Sexe',
    'region': 'Région',
    'departement_eff': 'Département'
}
len_dict = {}

# BOUCLE POUR DÉPENSES
for col in cols_to_calculate:
    if col == 'patho_niv1':
        # Exclure "Total"
        uniques = df_depenses.loc[
            ~df_depenses[col].isin(['Total', 'Total consommants tous régimes']),
            col
        ].dropna().unique()
    else:
        uniques = df_depenses[col].dropna().unique()
    
    len_dict[f"len_{col}"] = len(uniques) + 1

# BOUCLE POUR EFFECTIFS
for key_name, col_real_name in cols_to_calculate_effectifs.items():
    if col_real_name in df_effectifs.columns:
        if key_name == 'age':
            # Exclure 'tous âges', 'tous ages', 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
                col_real_name
            ].dropna().unique()
        elif key_name == 'patho_effectif':
            # Exclure 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['total']),
                col_real_name
            ].dropna().unique()
        else:
            uniques = df_effectifs[col_real_name].dropna().unique()
        
        len_dict[f"len_{key_name}"] = len(uniques) + 1

# Dépenses
annees = sorted(df_depenses['annee'].dropna().unique())

pathos = sorted(df_depenses.loc[
    ~df_depenses['patho_niv1'].isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

postes = sorted(df_depenses['dep_niv_1'].dropna().unique())
postes_niv2 = sorted(df_depenses['dep_niv_2'].dropna().unique())

# Effectifs
depts = sorted(df_effectifs['Département'].dropna().unique())

classes_age = sorted(df_effectifs.loc[
    ~df_effectifs['Classe d\'age'].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
    'Classe d\'age'
].dropna().unique())

sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())

pathos_eff = sorted(df_effectifs.loc[
    ~df_effectifs['patho_niv1'].astype(str).str.lower().isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

pprint.pprint(len_dict)

Création d'un onglet Filtres

Cet onglet est nécessaire pour alimenter les listes de validaiton dans les filtres.

Je vais récupérer l'année, la pathologie et le poste de dépense avec la fonction UNIQUE() depuis les onglets "cleaned_data"

In [ ]:
from openpyxl.utils import FORMULAE
"UNIQUE" in FORMULAE

In [ ]:
# DÉFINITION DES NOMS DE ONGLETS SOURCES
sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'

#CHARGEMENT DES WORKBOOKS
wb_eff = load_workbook(output_eff)
wb_dep = load_workbook(output_dep)

# GESTION DE L'ONGLET FILTRES
if 'Filtres' not in wb_eff.sheetnames:
    filtres_sheet = wb_eff.create_sheet('Filtres', 0)
else:
    filtres_sheet = wb_eff['Filtres']
    filtres_sheet.delete_rows(1, filtres_sheet.max_row)

pathologies_uniques = sorted([p for p in df_effectifs['patho_niv1'].unique() 
                              if p and "total" not in str(p).lower()])
for idx, patho in enumerate(['Tous', 'Total consommants tous régimes'] + pathologies_uniques):
    filtres_sheet[f'A{idx + 1}'] = patho

#les classes d'âge
classes_age_uniques = sorted([c for c in df_effectifs['Classe d\'age'].unique() if c])
for idx, age in enumerate(['Tous'] + classes_age_uniques):
    filtres_sheet[f'B{idx + 1}'] = age

# ARRAYFORMULAS EXCEL

# Formules basées sur les dépenses
formula_annee = f"=_xlfn.SORT(_xlfn.UNIQUE({sheet_dep}!A:A))"
filtres_sheet['A1'] = ArrayFormula(f"A1:A{len_dict['len_annee']}", formula_annee)

formula_patho = f"=_xlfn.UNIQUE({sheet_dep}!B:B)"
filtres_sheet['B1'] = ArrayFormula(f"B1:B{len_dict['len_patho_niv1']}", formula_patho)

formula_poste = f"=_xlfn.UNIQUE({sheet_dep}!E:E)"
filtres_sheet['C1'] = ArrayFormula(f"C1:C{len_dict['len_dep_niv_1']}", formula_poste)

formula_poste_niv2 = f"=_xlfn.UNIQUE({sheet_dep}!F:F)"
filtres_sheet['D1'] = ArrayFormula(f"D1:D{len_dict.get('len_dep_niv_1', len_dict['len_dep_niv_1'])}", formula_poste_niv2)

# Formules basées sur les effectifs
formula_annee_eff = f"=_xlfn.UNIQUE({sheet_eff}!A:A)"
filtres_sheet['E1'] = ArrayFormula(f"E1:E{len_dict.get('len_annee_eff', len_dict['len_annee_eff'])}", formula_annee_eff)

formula_patho_eff = f"=_xlfn.UNIQUE({sheet_eff}!B:B)"
filtres_sheet['F1'] = ArrayFormula(f"F1:F{len_dict.get('len_patho_effectif', len_dict['len_patho_effectif'])}", formula_patho_eff)

formula_age = f"=_xlfn.UNIQUE({sheet_eff}!H:H)"
filtres_sheet['G1'] = ArrayFormula(f"G1:G{len_dict.get('len_age', len_dict['len_age'])}", formula_age)

formula_sexe = f"=_xlfn.UNIQUE({sheet_eff}!I:I)"
filtres_sheet['H1'] = ArrayFormula(f"H1:H{len_dict['len_sexe']}", formula_sexe)

formula_region = f"=_xlfn.UNIQUE({sheet_eff}!J:J)"
filtres_sheet['I1'] = ArrayFormula(f"I1:I{len_dict['len_region']}", formula_region)

formula_dept = f"=_xlfn.UNIQUE({sheet_eff}!K:K)"
filtres_sheet['J1'] = ArrayFormula(f"J1:J{len_dict['len_departement_eff']}", formula_dept)
dv_patho = DataValidation(type='list', formula1=f"=Filtres!$A$1:$A${len(pathologies_uniques) + 2}")
wb_eff['Effectifs'].add_data_validation(dv_patho)
dv_patho.add('B4')

dv_age = DataValidation(type='list', formula1=f"=Filtres!$B$1:$B${len(classes_age_uniques) + 1}")
wb_eff['Effectifs'].add_data_validation(dv_age)
dv_age.add('B5')

## Fonctions utilitaires

Comme nous allons répéter cette opération plusieurs fois, il est préférable de créer une fonction qui effectue les mêmes opérations.

In [ ]:
def auto_adjust_columns(sheet: Worksheet) -> None:
    """Ajuste automatiquement la largeur des colonnes selon le contenu."""
    for col_idx, column in enumerate(sheet.columns, 1):
        max_length = 0
        col_letter = get_column_letter(col_idx)
        for cell in column:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        # max_length + 2, plafonné à 50
        adjusted_width = min(max_length + 2, 50)
        sheet.column_dimensions[col_letter].width = max(adjusted_width, 12)


def add_filter_title(sheet: Worksheet, title: str, start_row: int, start_column: int) -> None:
    """Ajoute le titre du filtre avec la mise en forme."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = title
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid') 
    cell.font = Font(bold=True)


def add_filter_value(sheet: Worksheet, value, start_row: int, start_column: int) -> None:
    """Ajoute la valeur par défaut du filtre."""
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = value
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid')
    cell.font = Font(bold=True)


def add_data_validation(sheet: Worksheet, start_row: int, start_column: int, formula: str) -> None:
    """Ajoute la liste déroulante (Data Validation) sur la cellule cible."""
    dv = DataValidation(type='list', formula1=formula)
    sheet.add_data_validation(dv)
    dv.add(sheet.cell(row=start_row, column=start_column).coordinate)


def add_filter(sheet: Worksheet, title: str, value, start_row: int, start_column: int, end_row: int, end_column: int, formula: str) -> None:
    """Construit un bloc filtre complet avec cellules fusionnées et liste déroulante."""
    target_val_column = end_column + 1
    
    add_filter_title(sheet, title, start_row, start_column)
    add_filter_value(sheet, value, start_row, target_val_column)
    add_data_validation(sheet, start_row, target_val_column, formula)
    
    sheet.merge_cells(start_row=start_row, start_column=start_column, end_row=end_row, end_column=end_column)
    sheet.merge_cells(start_row=start_row, start_column=target_val_column, end_row=end_row, end_column=target_val_column + 1)

# Charger les deux fichiers séparés
wb_dep = load_workbook(output_dep)
wb_eff = load_workbook(output_eff)

# Sauvegarde indépendante des deux classeurs 
wb_eff.save(output_eff)

wb_dep.close()
wb_eff.close()

In [ ]:
from openpyxl.worksheet.datavalidation import DataValidation

wb_eff = load_workbook(output_eff)
wb_dep = load_workbook(output_dep)

if 'Filtres' not in wb_eff.sheetnames:
    filtres_sheet = wb_eff.create_sheet('Filtres', 0)
else:
    filtres_sheet = wb_eff['Filtres']
    filtres_sheet.delete_rows(1, filtres_sheet.max_row)

filtres_sheet.sheet_view.showGridLines = False

pathologies_uniques = sorted([p for p in df_effectifs['patho_niv1'].unique() 
                              if p and "total" not in str(p).lower()])
pathologies_list = ['Tous', 'Total consommants tous régimes'] + pathologies_uniques

for idx, patho in enumerate(pathologies_list):
    filtres_sheet[f'A{idx + 1}'] = patho
num_pathologies = len(pathologies_list)

classes_age_uniques = sorted([c for c in df_effectifs['Classe d\'age'].unique() if c])
classes_age_list = ['Tous'] + classes_age_uniques

for idx, age in enumerate(classes_age_list):
    filtres_sheet[f'B{idx + 1}'] = age
num_ages = len(classes_age_list)

filtres_sheet.cell(row=2, column=3, value=f"Classes d'âge: {num_ages}")

dv_patho = DataValidation(
    type='list',
    formula1=f"=Filtres!$A$1:$A${num_pathologies}"
)
wb_eff['Effectifs'].add_data_validation(dv_patho)
dv_patho.add('B4')

dv_age = DataValidation(
    type='list',
    formula1=f"=Filtres!$B$1:$B${num_ages}"
)
wb_eff['Effectifs'].add_data_validation(dv_age)
dv_age.add('B5')

wb_eff.save(output_eff)
wb_dep.save(output_dep)
wb_eff.close()
wb_dep.close()

Création du dashboard

Je vais créer un premier tableau de bord interactif avec des filtres.
Je vais me concentrer sur les dépenses de remboursement par pathologie et par poste de dépense.
L'utilisateur aura la possibilité de choisir l'année, la pathologie et le poste de dépense via des filtres sous forme de listes de validation.
Ces filtres permettront d'actualiser dynamiquement les formules et les graphiques.

Design :

In [ ]:
wb_eff = load_workbook(output_eff)
wb_dep = load_workbook(output_dep)
# === COULEURS ===
COULEURS = {
    'bleu_fonce': 'FF1F4E78',
    'bleu_clair': 'FF4472C4',
    'cyan': 'FF00CCFF',
    'jaune_filtre': 'FFF2CC',
    'gris_clair': 'FFCCCCCC',
}

# Noms des sheets d'origine
sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'



## Onglet postes

In [ ]:
wb_dep = load_workbook(output_dep)

# ONGLET POSTES
if 'Postes' not in wb_dep.sheetnames:
    postes_sheet = wb_dep.create_sheet('Postes', 1)
else:
    postes_sheet = wb_dep['Postes']
    postes_sheet.delete_rows(1, postes_sheet.max_row)

postes_sheet.sheet_view.showGridLines = False

# === 2. FILTRES DE SÉLECTION (ANNÉE + POSTE) ===
postes_sheet['A1'] = 'Année'
postes_sheet['A1'].font = Font(bold=True, color='FFFFFF')
postes_sheet['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')

postes_sheet['B1'] = int(annees[-1])
postes_sheet['B1'].font = Font(bold=True)
postes_sheet['B1'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')

dv_annee_postes = DataValidation(type='list', formula1=f"=Filtres!$A$2:$A${len_dict['len_annee']}")
wb_dep['Postes'].add_data_validation(dv_annee_postes)
dv_annee_postes.add('B1')

postes_sheet['A2'] = 'Poste'
postes_sheet['A2'].font = Font(bold=True, color='FFFFFF')
postes_sheet['A2'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')

postes_sheet['B2'] = "Tous"
postes_sheet['B2'].font = Font(bold=True)
postes_sheet['B2'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')

postes_sheet['F2'] = f"=UNIQUE(FILTER('{sheet_dep}'!E:E, '{sheet_dep}'!A:A=B$1))"
dv_poste = DataValidation(type='list', formula1="=F2:F100")
wb_dep['Postes'].add_data_validation(dv_poste)
dv_poste.add('B2')

postes_sheet.column_dimensions['F'].hidden = True

postes_sheet.merge_cells('A3:E5')
postes_sheet['A3'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')

postes_sheet['A7'] = "Total dépenses"
postes_sheet['A7'].font = Font(bold=True, size=11)

postes_sheet['B7'] = f"=IFERROR(IF(B$2=\"Tous\", SUM(B11:B{50}), SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, B$1, '{sheet_dep}'!E:E, B$2)), 0)"
postes_sheet['B7'].font = Font(bold=True, size=14, color=COULEURS['bleu_fonce'])
postes_sheet['B7'].number_format = '#,##0'

postes_sheet['A9'] = "Poste"
postes_sheet['B9'] = "Dépenses"
postes_sheet['C9'] = "Patients"
postes_sheet['D9'] = "Coût/patient"

for col in ['A', 'B', 'C', 'D']:
    postes_sheet[f'{col}9'].font = Font(bold=True, color='FFFFFF')
    postes_sheet[f'{col}9'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row_poste = 11
postes_valides = [p for p in postes if p and "total" not in p.lower()]

for poste in postes_valides:
    postes_sheet[f'A{row_poste}'] = poste
    
    postes_sheet[f'B{row_poste}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, B$1, '{sheet_dep}'!E:E, A{row_poste}), 0)"
    postes_sheet[f'B{row_poste}'].number_format = '#,##0'
    
    postes_sheet[f'C{row_poste}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!H:H, '{sheet_dep}'!A:A, B$1, '{sheet_dep}'!E:E, A{row_poste}), 0)"
    postes_sheet[f'C{row_poste}'].number_format = '#,##0'
    
    postes_sheet[f'D{row_poste}'] = f"=IFERROR(B{row_poste} / C{row_poste}, 0)"
    postes_sheet[f'D{row_poste}'].number_format = '#,##0'
    
    row_poste += 1

postes_sheet['B7'] = f"=IFERROR(IF(B$2=\"Tous\", SUM(B11:B{row_poste-1}), SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, B$1, '{sheet_dep}'!E:E, B$2)), 0)"

auto_adjust_columns(postes_sheet)

wb_dep.save(output_dep)
wb_dep.close()

## Onglet depenses

In [ ]:
# Chargement du classeur
wb_dep = load_workbook(output_dep)
nom_feuille_source = 'Depenses' 

if 'Depenses' in wb_dep.sheetnames:
    depenses_sheet = wb_dep['Depenses']
    
    # Nettoyage
    for row in range(6, 150):
        for col in ['A', 'B', 'C', 'D', 'E', 'F']:
            depenses_sheet[f'{col}{row}'] = None

    # Formules avec la source correcte
    prefixe = f"'{source_data}'!" 
    
    # Construction des conditions avec les BONNES colonnes
    # Colonne A = année, Colonne E = dep_niv_1 (pathologie)
    # Filtre année en $C$1, filtre pathologie en $C$3
    condition_annee = f"({prefixe}$A:$A=$C$1)"
    condition_poste = f"IF($C$3=\"Tous\", 1, {prefixe}$E:$E=$C$3)"
    
    # Insertion des formules
    formule_filter = f"FILTER({prefixe}$E:$E, {condition_annee} * {condition_poste}, \"Aucune donnée\")"
    depenses_sheet['A9'] = f"=SORT(UNIQUE({formule_filter}))"

    # B9 : Dépenses (colonne G)
    depenses_sheet['B9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS({prefixe}$G:$G, "
        f"{prefixe}$A:$A, $C$1, "
        f"{prefixe}$E:$E, A9#))"
    )
    
    # C9 : Patients/Ntop (colonne H)
    depenses_sheet['C9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS({prefixe}$H:$H, "
        f"{prefixe}$A:$A, $C$1, "
        f"{prefixe}$E:$E, A9#))"
    )
    
    # D9 : Coût/patient
    depenses_sheet['D9'] = f"=IF(C9#=0, 0, B9#/C9#)"

    # Styles et formatage
    for row in range(9, 100):
        depenses_sheet[f'B{row}'].number_format = '#,##0 €'
        depenses_sheet[f'C{row}'].number_format = '#,##0'
        depenses_sheet[f'D{row}'].number_format = '#,##0.00 €'

    # Entêtes et totaux
    depenses_sheet['A6'] = "Total dépenses :"
    depenses_sheet['A6'].font = Font(bold=True, size=11)
    
    depenses_sheet['B6'] = "=SUM(B9#)"
    depenses_sheet['B6'].font = Font(bold=True, size=12, color="1F497D")
    depenses_sheet['B6'].number_format = '#,##0 €'

    depenses_sheet['A8'] = "Pathologie"
    depenses_sheet['B8'] = "Dépenses"
    depenses_sheet['C8'] = "Patients"
    depenses_sheet['D8'] = "Coût/patient"

    for col in ['A', 'B', 'C', 'D']:
        cell = depenses_sheet[f'{col}8']
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
        depenses_sheet.column_dimensions[col].width = 30 if col != 'A' else 55    

    # Finalisation
    wb_dep.save(output_dep)
    wb_dep.close()
else:
    print("Liste des feuilles :", wb_dep.sheetnames)

## Onglet Effectifs

In [ ]:

wb_eff = load_workbook(output_eff)

if 'Effectifs' in wb_eff.sheetnames:
    effectifs_sheet = wb_eff['Effectifs']
    
    # Nettoyer proprement TOUS les anciens graphiques de l'onglet
    effectifs_sheet._charts.clear()
    

    for row in range(5, 40):
        effectifs_sheet[f'J{row}'] = None
        effectifs_sheet[f'K{row}'] = None
        for col_idx in range(20, 38): # Colonnes T à AK
            effectifs_sheet.cell(row=row, column=col_idx, value=None)

    # 1. CRÉATION DU FILTRE DE PATHOLOGIE VISUEL (Zone B4)
    effectifs_sheet['A4'] = "Filtrer par Pathologie :"
    effectifs_sheet['A4'].font = Font(bold=True, color="000000")
    
    effectifs_sheet['B4'] = "Total consommants tous régimes" 
    effectifs_sheet['B4'].font = Font(bold=True, color="1F497D")
    effectifs_sheet['B4'].fill = PatternFill(start_color="E9EDF4", fill_type="solid")

    patho_filtre = effectifs_sheet['B4'].value
    
    # Application du filtre selon la sélection
    if patho_filtre and patho_filtre != "Total consommants tous régimes":
        df_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() == patho_filtre.strip()]
    else:
        df_filtered = df_effectifs[df_effectifs['patho_niv1'].str.strip() != "Total consommants tous régimes"]

    df_age = df_filtered.groupby('Classe d\'age')['Effectif'].sum().reset_index()
    # Tri décroissant pour faire ressortir le Top des âges
    df_age = df_age.sort_values(by='Effectif', ascending=False)

    effectifs_sheet['A6'] = "Classe d'âge"
    effectifs_sheet['B6'] = "Effectif"
    for col in ['A', 'B']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_age = 7
    for _, r in df_age.iterrows():
        effectifs_sheet[f'A{row_age}'] = r['Classe d\'age']
        effectifs_sheet[f'B{row_age}'] = r['Effectif']
        effectifs_sheet[f'B{row_age}'].number_format = '#,##0'
        row_age += 1
        
    for r_clear in range(row_age, 30):
        effectifs_sheet[f'A{r_clear}'] = None
        effectifs_sheet[f'B{r_clear}'] = None

    df_sexe = df_filtered.groupby('Sexe')['Effectif'].sum().reset_index()
    
    effectifs_sheet['D6'] = "Sexe"
    effectifs_sheet['E6'] = "Effectif"
    for col in ['D', 'E']:
        effectifs_sheet[f'{col}6'].font = Font(bold=True, color='FFFFFF')
        effectifs_sheet[f'{col}6'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_sexe = 7
    for _, r in df_sexe.iterrows():
        effectifs_sheet[f'D{row_sexe}'] = r['Sexe']
        effectifs_sheet[f'E{row_sexe}'] = r['Effectif']
        effectifs_sheet[f'E{row_sexe}'].number_format = '#,##0'
        row_sexe += 1
        
    for r_clear in range(row_sexe, 15):
        effectifs_sheet[f'D{r_clear}'] = None
        effectifs_sheet[f'E{r_clear}'] = None

    chart_age = BarChart()
    chart_age.type = "col"
    chart_age.style = 10
    chart_age.title = f"Top 10 Classes d'âge - {patho_filtre if patho_filtre else 'Global'}"
    chart_age.y_axis.title = "Effectifs"
    chart_age.x_axis.title = "Classes d'age"
    
    max_row_graph = min(row_age - 1, 16)
    
    data_age = Reference(effectifs_sheet, min_col=2, min_row=6, max_row=max_row_graph) # Colonne B (Effectifs)
    cats_age = Reference(effectifs_sheet, min_col=1, min_row=7, max_row=max_row_graph) # Colonne A (Ages)
    
    chart_age.add_data(data_age, titles_from_data=True)
    chart_age.set_categories(cats_age)
    chart_age.legend = None # Pas besoin de légende pour une seule série
    
    chart_age.height = 14
    chart_age.width = 20
    effectifs_sheet.add_chart(chart_age, "A22")

    auto_adjust_columns(effectifs_sheet)

wb_eff.save(output_eff)
wb_eff.close()

Dépenses

In [ ]:
# Chargement du classeur
wb_dep = load_workbook(output_dep)
source_data = 'cleanedData_depenses'

ws_source = wb_dep[source_data] 
max_src = ws_source.max_row

if 'Depenses' in wb_dep.sheetnames:
    depenses_sheet = wb_dep['Depenses']
    
    # Nettoyage
    for row in range(6, 150):
        for col in ['A', 'B', 'C', 'D', 'E', 'F']:
            depenses_sheet[f'{col}{row}'] = None

    prefixe = f"'{source_data}'!" 
    

    condition_annee = f"({prefixe}$A:$A=$C$1)"
    condition_poste = f"IF($C$3=\"Tous\", 1, {prefixe}$E:$E=$C$3)"
    
    # Insertion des formules
    formule_filter = f"FILTER({prefixe}$E:$E, {condition_annee} * {condition_poste}, \"Aucune donnée\")"
    depenses_sheet['A9'] = f"=SORT(UNIQUE({formule_filter}))"

    # B9 : Dépenses (colonne G)
    depenses_sheet['B9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS({prefixe}$G:$G, "
        f"{prefixe}$A:$A, $C$1, "
        f"{prefixe}$E:$E, A9#))"
    )
    
    # C9 : Patients/Ntop (colonne H)
    depenses_sheet['C9'] = (
        f"=IF(A9#=\"\", \"\", SUMIFS({prefixe}$H:$H, "
        f"{prefixe}$A:$A, $C$1, "
        f"{prefixe}$E:$E, A9#))"
    )
    
    # D9 : Coût/patient
    depenses_sheet['D9'] = f"=IF(C9#=0, 0, B9#/C9#)"

    # Styles et formatage
    for row in range(9, 100):
        depenses_sheet[f'B{row}'].number_format = '#,##0 €'
        depenses_sheet[f'C{row}'].number_format = '#,##0'
        depenses_sheet[f'D{row}'].number_format = '#,##0.00 €'

    # Entêtes et totaux
    depenses_sheet['A6'] = "Total dépenses :"
    depenses_sheet['A6'].font = Font(bold=True, size=11)
    
    depenses_sheet['B6'] = "=SUM(B9#)"
    depenses_sheet['B6'].font = Font(bold=True, size=12, color="1F497D")
    depenses_sheet['B6'].number_format = '#,##0 €'

    depenses_sheet['A8'] = "Pathologie"
    depenses_sheet['B8'] = "Dépenses"
    depenses_sheet['C8'] = "Patients"
    depenses_sheet['D8'] = "Coût/patient"

    for col in ['A', 'B', 'C', 'D']:
        cell = depenses_sheet[f'{col}8']
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
        depenses_sheet.column_dimensions[col].width = 30 if col != 'A' else 55    

    # Finalisation
    wb_dep.save(output_dep)
    wb_dep.close()

Les graphiques :

In [ ]:
# Graphique pathos dans depenses
wb_eff = load_workbook(output_eff)

chart2 = PieChart()
chart2.title = "Distribution par pathologie"
chart2.height = 12
chart2.width = 18
if row_patho > 9:
    chart2.add_data(Reference(pathos_sheet, min_col=2, min_row=9, max_row=row_patho-1), titles_from_data=False)
    chart2.set_categories(Reference(pathos_sheet, min_col=1, min_row=9, max_row=row_patho-1))
    pathos_sheet.add_chart(chart2, "H5")
wb_dep.save(output_dep)
wb_dep.close()

In [ ]:
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.series import SeriesLabel

wb_dep = load_workbook(output_dep)

if 'Postes' in wb_dep.sheetnames:
    postes_sheet = wb_dep['Postes']
    
    row_poste = 9
    while postes_sheet[f'A{row_poste}'].value is not None:
        row_poste += 1
        
    chart_postes = BarChart()
    chart_postes.type = "bar"
    chart_postes.title = "Distribution par poste de dépenses"
    chart_postes.height = 12
    chart_postes.width = 18
    
    data_postes = Reference(postes_sheet, min_col=2, min_row=9, max_row=row_poste-1)
    cats_postes = Reference(postes_sheet, min_col=1, min_row=9, max_row=row_poste-1)
    
    chart_postes.add_data(data_postes, titles_from_data=False)
    chart_postes.set_categories(cats_postes)
    
    if chart_postes.series:
        chart_postes.series[0].title = SeriesLabel(v="Dépenses")
        
    postes_sheet.add_chart(chart_postes, "G5")
    auto_adjust_columns(postes_sheet)

if 'Pathologies' in wb_dep.sheetnames:
    pathos_sheet = wb_dep['Pathologies']
    
    row_patho = 9
    while pathos_sheet[f'A{row_patho}'].value is not None:
        row_patho += 1
        
    if row_patho > 9:
        chart_pathos = PieChart()
        chart_pathos.title = "Distribution des dépenses par pathologie"
        chart_pathos.height = 12
        chart_pathos.width = 18
        
        data_pathos = Reference(pathos_sheet, min_col=2, min_row=9, max_row=row_patho-1)
        cats_pathos = Reference(pathos_sheet, min_col=1, min_row=9, max_row=row_patho-1)
        
        chart_pathos.add_data(data_pathos, titles_from_data=False)
        chart_pathos.set_categories(cats_pathos)
        
        if chart_pathos.series:
            chart_pathos.series[0].title = SeriesLabel(v="Dépenses")
            
        pathos_sheet.add_chart(chart_pathos, "H5")
    
    auto_adjust_columns(pathos_sheet)

wb_dep.save(output_dep)
wb_dep.close()

Effectifs :

In [ ]:
wb_eff = load_workbook(output_eff)

if 'Effectifs' in wb_eff.sheetnames:
    effectifs_sheet = wb_eff['Effectifs']
    effectifs_sheet.delete_rows(1, effectifs_sheet.max_row)
    effectifs_sheet._charts.clear()
    effectifs_sheet.sheet_view.showGridLines = False

    effectifs_sheet.merge_cells('C1:H2')
    effectifs_sheet['C1'] = "Diagnostic des pathologies\n2022 - 2023"
    effectifs_sheet['C1'].font = Font(bold=True, size=16, color='FFFFFF')
    effectifs_sheet['C1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
    effectifs_sheet['C1'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    effectifs_sheet.row_dimensions[1].height = 30

    effectifs_sheet['A4'] = "Pathologie"
    effectifs_sheet['A4'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet['A4'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
    effectifs_sheet['B4'] = "Tous"
    effectifs_sheet['B4'].font = Font(bold=True)
    effectifs_sheet['B4'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')

    pathologies_uniques = sorted([p for p in df_effectifs['patho_niv1'].unique() 
                                 if p and "total" not in str(p).lower()])
    dv_patho = DataValidation(type='list', formula1=f"Tous,{','.join(pathologies_uniques)}")
    effectifs_sheet.add_data_validation(dv_patho)
    dv_patho.add('B4')

    effectifs_sheet['A5'] = "Classe d'âge"
    effectifs_sheet['A5'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet['A5'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
    effectifs_sheet['B5'] = "Tous"
    effectifs_sheet['B5'].font = Font(bold=True)
    effectifs_sheet['B5'].fill = PatternFill(start_color=COULEURS['jaune_filtre'], fill_type='solid')

    classes_age_uniques = sorted([c for c in df_effectifs['Classe d\'age'].unique() if c])
    dv_age = DataValidation(type='list', formula1=f"Tous,{','.join(classes_age_uniques)}")
    effectifs_sheet.add_data_validation(dv_age)
    dv_age.add('B5')

    patho_value = effectifs_sheet['B4'].value
    age_value = effectifs_sheet['B5'].value

    df_base = df_effectifs[df_effectifs['patho_niv1'].str.strip() != "Total consommants tous régimes"]

    if patho_value and patho_value != "Tous":
        df_filtered = df_base[df_base['patho_niv1'].str.strip() == patho_value.strip()]
    else:
        df_filtered = df_base

    if age_value and age_value != "Tous":
        df_filtered = df_filtered[df_filtered['Classe d\'age'].str.strip() == age_value.strip()]

    pivot_sexe = df_filtered[df_filtered['Sexe'].str.strip() != 'tous sexes'].pivot_table(
        index='patho_niv1',
        columns='Sexe',
        values='Effectif',
        aggfunc='sum',
        fill_value=0
    )

    effectifs_sheet['A7'] = "Pathologie"
    effectifs_sheet['A7'].font = Font(bold=True, color='FFFFFF')
    effectifs_sheet['A7'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')
    effectifs_sheet.column_dimensions['A'].width = 55

    for col_idx, sexe in enumerate(pivot_sexe.columns):
        cell = effectifs_sheet.cell(row=7, column=2 + col_idx, value=sexe)
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

    row_sexe = 8
    all_values = []
    
    for patho, row_data in pivot_sexe.iterrows():
        effectifs_sheet.cell(row=row_sexe, column=1, value=patho)
        for col_idx, value in enumerate(row_data):
            cell = effectifs_sheet.cell(row=row_sexe, column=2 + col_idx, value=int(value))
            cell.number_format = '#,##0'
            all_values.append(int(value))
        row_sexe += 1

    last_row_sexe = row_sexe - 1

    if all_values:
        moyenne = sum(all_values) / len(all_values)
        red_font = Font(bold=True, color='FF0000')
        red_fill = PatternFill(start_color='FFFF00', fill_type='solid')
        
        for row in range(8, last_row_sexe + 1):
            for col in range(2, 2 + len(pivot_sexe.columns)):
                cell_ref = effectifs_sheet.cell(row=row, column=col)
                effectifs_sheet.conditional_formatting.add(f'{cell_ref.coordinate}',
                                                          CellIsRule(operator='greaterThanOrEqual',
                                                                     formula=[str(int(moyenne))],
                                                                     font=red_font,
                                                                     fill=red_fill))

    chart = BarChart()
    chart.type = "bar"
    chart.grouping = "percentStacked"
    chart.title = f"Effectifs par Classe d'âge - {patho_value if patho_value != 'Tous' else 'Global'}"
    chart.style = 12
    chart.height = 18
    chart.width = 26

    data = Reference(
        effectifs_sheet,
        min_col=2,
        min_row=7,
        max_row=last_row_sexe,
        max_col=1 + len(pivot_sexe.columns)
    )
    chart.add_data(data, titles_from_data=True)

    cats = Reference(effectifs_sheet, min_col=1, min_row=8, max_row=last_row_sexe)
    chart.set_categories(cats)

    chart.legend.position = 'r'
    chart.legend.include_in_layout = False
    chart.x_axis.scaling.min = 0
    chart.x_axis.scaling.max = 100

    effectifs_sheet.add_chart(chart, "A" + str(last_row_sexe + 3))

    auto_adjust_columns(effectifs_sheet)

    wb_eff.save(output_eff)
    wb_eff.close()

In [ ]:
if 'Départements' not in wb_eff.sheetnames:
    dept_sheet = wb_eff.create_sheet('Départements')
else:
    dept_sheet = wb_eff['Départements']
    dept_sheet.delete_rows(1, dept_sheet.max_row)

dept_sheet.sheet_view.showGridLines = False

dept_sheet.merge_cells('A1:H1')
dept_sheet['A1'] = "TOP 10 DES DÉPARTEMENTS"
dept_sheet['A1'].font = Font(bold=True, size=14, color='FFFFFF')
dept_sheet['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], fill_type='solid')
dept_sheet['A1'].alignment = Alignment(horizontal='center', vertical='center')
dept_sheet.row_dimensions[1].height = 25

top10_age = df_effectifs.groupby('Département')['Effectif'].sum().nlargest(10)

dept_sheet['A3'] = "Département"
dept_sheet['B3'] = "Effectif"
for col in ['A', 'B']:
    dept_sheet[f'{col}3'].font = Font(bold=True, color='FFFFFF')
    dept_sheet[f'{col}3'].fill = PatternFill(start_color=COULEURS['bleu_clair'], fill_type='solid')

row = 4
for dept, eff in top10_age.items():
    dept_sheet[f'A{row}'] = dept
    dept_sheet[f'B{row}'] = int(eff)
    dept_sheet[f'B{row}'].number_format = '#,##0'
    row += 1

last_row_dept = row - 1

chart1 = BarChart()
chart1.type = "col"
chart1.title = "Tranche d'âge"
chart1.style = 10
chart1.height = 10
chart1.width = 12

data1 = Reference(dept_sheet, min_col=2, min_row=3, max_row=last_row_dept)
cats1 = Reference(dept_sheet, min_col=1, min_row=4, max_row=last_row_dept)

chart1.add_data(data1, titles_from_data=True)
chart1.set_categories(cats1)
chart1.legend = None
dept_sheet.add_chart(chart1, "A" + str(last_row_dept + 2))

chart2 = BarChart()
chart2.type = "col"
chart2.title = "Sexe"
chart2.style = 10
chart2.height = 10
chart2.width = 12

chart2.add_data(data1, titles_from_data=True)
chart2.set_categories(cats1)
chart2.legend = None
dept_sheet.add_chart(chart2, "N" + str(last_row_dept + 2))

chart3 = BarChart()
chart3.type = "col"
chart3.title = "Département"
chart3.style = 10
chart3.height = 10
chart3.width = 12

chart3.add_data(data1, titles_from_data=True)
chart3.set_categories(cats1)
chart3.legend = None
dept_sheet.add_chart(chart3, "AA" + str(last_row_dept + 2))

auto_adjust_columns(dept_sheet)
wb_eff.save(output_eff)

## Sauvegarde finale

In [ ]:
# 1. Sauvegarde des classeurs individuels (chacun sa variable)
wb_dep.save(output_dep)
wb_eff.save(output_eff)

# 2. Fermeture des classeurs (les parenthèses doivent rester totalement vides)
wb_dep.close()
wb_eff.close()